# External Data Collection & Signal Engineering

**Bitcoin DCA Capstone — GT OMSA Practicum**

This notebook documents the collection and analysis of **external data sources** used in our novel 6-signal DCA strategy.

## Data Sources

| Source | Data | API | Frequency | Signal Use |
|--------|------|-----|-----------|------------|
| FRED (Federal Reserve) | Fed Funds Rate (DFF) | pandas_datareader | Daily | Monetary policy velocity |
| FRED (Federal Reserve) | M2 Money Supply (M2SL) | pandas_datareader | Monthly→Daily | Liquidity signal |
| FRED (Federal Reserve) | 10Y-2Y Yield Spread (T10Y2Y) | pandas_datareader | Daily | Recession indicator |
| Alternative.me | Crypto Fear & Greed Index | REST API (free) | Daily | Contrarian sentiment |
| Yahoo Finance / yfinance | VIX (^VIX) | yfinance | Daily | Market fear signal |
| Yahoo Finance / yfinance | SPY (S&P500) | yfinance | Daily | Risk-on/off regime |
| Yahoo Finance / yfinance | UUP (DXY proxy) | yfinance | Daily | Dollar strength |

## Novel Hypothesis

> **Monetary tightening** (Fed rate hikes + strong dollar + shrinking M2) is a systematic headwind for Bitcoin that the base MVRV model ignores. By reducing DCA weight during tightening cycles and increasing during easing, we can improve accumulation timing—especially during periods like the 2022 rate hike cycle.
>
> **Extreme fear** (VIX spikes + Crypto Fear & Greed at extremes) is a contrarian buy signal—panic selling creates the best long-term accumulation windows.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Project root
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (14, 5)

EXT = ROOT / "data" / "externals"
print(f"Project root: {ROOT}")
print(f"External data: {EXT}")

## 1. FRED Data — Macroeconomic Indicators

Downloaded via `pandas_datareader` — no API key required.

In [ ]:
# Load FRED data
dff = pd.read_csv(EXT / "fred" / "dff.csv", index_col="date", parse_dates=True)
m2  = pd.read_csv(EXT / "fred" / "m2sl.csv", index_col="date", parse_dates=True)
t10y2y = pd.read_csv(EXT / "fred" / "t10y2y.csv", index_col="date", parse_dates=True)

print("=== Fed Funds Rate (DFF) ===")
print(f"  Rows: {len(dff)}, Range: {dff.index.min().date()} ~ {dff.index.max().date()}")
print(f"  Min: {dff['dff'].min():.2f}%  Max: {dff['dff'].max():.2f}%  Current: {dff['dff'].iloc[-1]:.2f}%")
print()
print("=== M2 Money Supply (M2SL) ===")
print(f"  Rows: {len(m2)}, Range: {m2.index.min().date()} ~ {m2.index.max().date()}")
print(f"  Min: ${m2['m2sl'].min():,.0f}B  Max: ${m2['m2sl'].max():,.0f}B  Current: ${m2['m2sl'].iloc[-1]:,.0f}B")
print()
print("=== 10Y-2Y Yield Spread (T10Y2Y) ===")
print(f"  Rows: {len(t10y2y)}, Range: {t10y2y.index.min().date()} ~ {t10y2y.index.max().date()}")
print(f"  Min: {t10y2y['t10y2y'].min():.2f}%  Max: {t10y2y['t10y2y'].max():.2f}%  Current: {t10y2y['t10y2y'].iloc[-1]:.2f}%")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Fed Funds Rate
ax = axes[0]
ax.plot(dff.index, dff["dff"], color="#dc2626", linewidth=1.5)
ax.fill_between(dff.index, dff["dff"], 0, alpha=0.15, color="#dc2626")
ax.set_ylabel("Rate (%)", fontsize=10)
ax.set_title("Fed Funds Effective Rate (DFF) — FRED", fontsize=11, fontweight="bold")
ax.annotate("2022-2023\nRate Hike Cycle", xy=(pd.Timestamp("2023-01-01"), 4.3),
            xytext=(pd.Timestamp("2020-01-01"), 4.5),
            arrowprops=dict(arrowstyle="->", color="#dc2626"), fontsize=9, color="#dc2626")

# M2 Money Supply YoY
ax = axes[1]
m2_yoy = m2["m2sl"].pct_change(365) * 100
ax.plot(m2.index, m2_yoy, color="#2563eb", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax.fill_between(m2.index, m2_yoy, 0, where=(m2_yoy > 0), alpha=0.15, color="#2563eb", label="Expansion")
ax.fill_between(m2.index, m2_yoy, 0, where=(m2_yoy < 0), alpha=0.15, color="#dc2626", label="Contraction")
ax.set_ylabel("YoY Growth (%)", fontsize=10)
ax.set_title("M2 Money Supply YoY Growth (M2SL) — FRED", fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.annotate("COVID stimulus\n+26% YoY peak", xy=(pd.Timestamp("2021-02-01"), 26),
            xytext=(pd.Timestamp("2019-01-01"), 22),
            arrowprops=dict(arrowstyle="->", color="#2563eb"), fontsize=9, color="#2563eb")

# 10Y-2Y Spread (recession indicator)
ax = axes[2]
ax.plot(t10y2y.index, t10y2y["t10y2y"], color="#7c3aed", linewidth=1.5)
ax.axhline(0, color="#dc2626", linewidth=1.2, linestyle="--", label="Inversion threshold")
ax.fill_between(t10y2y.index, t10y2y["t10y2y"], 0,
                where=(t10y2y["t10y2y"] < 0), alpha=0.2, color="#dc2626", label="Inverted (recession risk)")
ax.set_ylabel("Spread (%)", fontsize=10)
ax.set_title("10Y-2Y Treasury Yield Spread — Recession Indicator (FRED)", fontsize=11, fontweight="bold")
ax.legend(fontsize=9)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
plt.suptitle("FRED Macroeconomic Data: Fed Policy + Liquidity + Yield Curve", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Key Observations — FRED Data

- **2022 rate hike cycle**: Fed raised rates from ~0% to 5.33% in 14 months — the fastest hiking pace since the 1980s. This is the primary driver of the 2022 BTC bear market.
- **COVID M2 expansion**: M2 grew 26% YoY in 2021 — unprecedented liquidity drove the crypto bull market. Our model captures this as a positive signal.
- **2022-2023 M2 contraction**: First time in decades M2 shrank YoY. Coincided with BTC dropping from $69k → $16k.
- **Yield curve inversion**: The 10Y-2Y spread went deeply negative in 2022-2023, historically a leading recession indicator.

## 2. Crypto Fear & Greed Index — Alternative.me

Free public REST API. No key required. Available from 2018-02-01.

In [ ]:
fg = pd.read_csv(EXT / "fear_greed" / "fear_greed.csv", index_col="date", parse_dates=True)

print("=== Crypto Fear & Greed Index ===")
print(f"  Rows: {len(fg)}, Range: {fg.index.min().date()} ~ {fg.index.max().date()}")
print(f"  Scale: 0 = Extreme Fear, 100 = Extreme Greed")
print()
print("Distribution by classification:")
print(fg["classification"].value_counts())
print()
print("Recent readings:")
print(fg.tail(7).to_string())

In [ ]:
# Load BTC price for comparison
from template.prelude_template import load_data
btc = load_data()[["PriceUSD_coinmetrics"]].rename(columns={"PriceUSD_coinmetrics": "btc_price"})

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

start = "2018-02-01"
fg_plot = fg.loc[start:]
btc_plot = btc.loc[start:]

# BTC Price
ax1.semilogy(btc_plot.index, btc_plot["btc_price"], color="#f59e0b", linewidth=1.5)
ax1.set_ylabel("BTC Price (log)", fontsize=10)
ax1.set_title("BTC Price", fontsize=11, fontweight="bold")

# Fear & Greed raw
cmap_colors = ["#dc2626" if v < 25 else "#f97316" if v < 45 else "#6b7280" if v < 55 else "#84cc16" if v < 75 else "#16a34a"
               for v in fg_plot["fear_greed"]]
ax2.bar(fg_plot.index, fg_plot["fear_greed"], color=cmap_colors, width=1, alpha=0.7)
ax2.axhline(25, color="#dc2626", linestyle="--", linewidth=0.8, label="Extreme Fear (25)")
ax2.axhline(75, color="#16a34a", linestyle="--", linewidth=0.8, label="Extreme Greed (75)")
ax2.axhline(50, color="black", linestyle=":", linewidth=0.8)
ax2.set_ylabel("Fear & Greed Score", fontsize=10)
ax2.set_title("Crypto Fear & Greed Index (Alternative.me) — 0=Extreme Fear, 100=Extreme Greed", fontsize=11, fontweight="bold")
ax2.legend(fontsize=9)
ax2.set_ylim(0, 100)

# Signal transformation (cubic)
fg_norm = (50 - fg_plot["fear_greed"]) / 50.0
fg_signal = fg_norm ** 3
ax3.plot(fg_plot.index, fg_signal, color="#be185d", linewidth=1.2)
ax3.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax3.fill_between(fg_plot.index, fg_signal, 0, where=(fg_signal > 0), alpha=0.2, color="#be185d", label="Buy signal (fear)")
ax3.fill_between(fg_plot.index, fg_signal, 0, where=(fg_signal < 0), alpha=0.2, color="#7c2d12", label="Reduce signal (greed)")
ax3.set_ylabel("Cubic Signal", fontsize=10)
ax3.set_title("Fear & Greed → Cubic Signal (emphasizes extremes, mutes middle)", fontsize=11, fontweight="bold")
ax3.legend(fontsize=9)

axes_list = [ax1, ax2, ax3]
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax3.xaxis.set_major_locator(mdates.YearLocator())
plt.suptitle("Crypto Fear & Greed Index: Raw Data + Signal Transformation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Why cubic? Show transformation comparison
fig, ax = plt.subplots(figsize=(10, 5))

x = np.linspace(0, 100, 200)  # Fear & Greed values
fg_norm_x = (50 - x) / 50.0

linear  = fg_norm_x                 # linear
cubic   = fg_norm_x ** 3            # cubic (our model)
squared = np.sign(fg_norm_x) * fg_norm_x ** 2  # squared with sign

ax.plot(x, linear,  label="Linear (naive)", color="#dc2626", linestyle="--", linewidth=2)
ax.plot(x, cubic,   label="Cubic (our model)", color="#059669", linewidth=2.5)
ax.plot(x, squared, label="Signed Square", color="#6366f1", linestyle="-.", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax.axvline(50, color="#6b7280", linewidth=0.8, linestyle=":")

# Annotate key points
for fg_val, label in [(10, "Extreme Fear"), (25, "Fear"), (75, "Greed"), (95, "Extreme Greed")]:
    norm = (50 - fg_val) / 50.0
    sig_c = norm ** 3
    sig_l = norm
    ax.annotate(f"FG={fg_val}\nCubic: {sig_c:+.2f}\nLinear: {sig_l:+.2f}",
                (fg_val, sig_c), textcoords="offset points", xytext=(5, 10 if sig_c > 0 else -30),
                fontsize=8, color="#059669")

ax.set_xlabel("Fear & Greed Score (0=Extreme Fear, 100=Extreme Greed)", fontsize=11)
ax.set_ylabel("Signal Strength", fontsize=11)
ax.set_title("Signal Transformation Comparison: Linear vs Cubic\n"
             "Cubic design: mutes moderate greed (75→-0.13), amplifies extreme greed (95→-0.73)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

print("Design rationale:")
print("  - Linear at FG=75 (Greed) → signal=-0.50 → would over-suppress buying in 2021 bull market")
print("  - Cubic at FG=75 (Greed)  → signal=-0.13 → muted, allows normal accumulation")
print("  - Cubic at FG=10 (E.Fear) → signal=+0.51 → strong contrarian buy (March 2020 COVID)")
print("  - Cubic at FG=95 (E.Greed)→ signal=-0.73 → strongly reduces at peak mania")

## 3. Market Volatility & Dollar Index — yfinance

VIX (fear gauge), SPY (S&P 500), UUP (DXY proxy — US Dollar Index).

In [ ]:
vix = pd.read_csv(EXT / "market" / "vix.csv", index_col="date", parse_dates=True)
spy = pd.read_csv(EXT / "market" / "spy.csv", index_col="date", parse_dates=True)
dxy = pd.read_csv(EXT / "market" / "dxy.csv", index_col="date", parse_dates=True)

print("=== VIX (CBOE Volatility Index) ===")
print(f"  Rows: {len(vix)}, Range: {vix.index.min().date()} ~ {vix.index.max().date()}")
print(f"  Min: {vix['vix'].min():.1f}  Max: {vix['vix'].max():.1f}  Current: {vix['vix'].iloc[-1]:.1f}")
print()
print("=== SPY (S&P 500 ETF) ===")
print(f"  Rows: {len(spy)}, Range: {spy.index.min().date()} ~ {spy.index.max().date()}")
print(f"  Min: ${spy['spy'].min():.1f}  Max: ${spy['spy'].max():.1f}  Current: ${spy['spy'].iloc[-1]:.1f}")
print()
print("=== UUP (DXY Proxy — US Dollar Strength) ===")
print(f"  Rows: {len(dxy)}, Range: {dxy.index.min().date()} ~ {dxy.index.max().date()}")
print(f"  Min: ${dxy['dxy'].min():.2f}  Max: ${dxy['dxy'].max():.2f}  Current: ${dxy['dxy'].iloc[-1]:.2f}")

In [ ]:
start = "2018-01-01"
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# BTC Price
ax = axes[0]
btc_plot = btc.loc[start:]
ax.semilogy(btc_plot.index, btc_plot["btc_price"], color="#f59e0b", linewidth=1.5)
ax.set_ylabel("BTC (log)", fontsize=10)
ax.set_title("BTC Price", fontsize=11, fontweight="bold")

# VIX
ax = axes[1]
vix_plot = vix.loc[start:]
ax.plot(vix_plot.index, vix_plot["vix"], color="#dc2626", linewidth=1.2)
ax.axhline(20, color="#6b7280", linestyle="--", linewidth=0.8, label="Normal (20)")
ax.axhline(30, color="#f59e0b", linestyle="--", linewidth=0.8, label="Elevated Fear (30)")
ax.axhline(45, color="#dc2626", linestyle="--", linewidth=0.8, label="Extreme Fear (45)")
ax.fill_between(vix_plot.index, vix_plot["vix"], 30, where=(vix_plot["vix"] > 30),
                alpha=0.2, color="#dc2626", label="High fear zone")
ax.set_ylabel("VIX", fontsize=10)
ax.set_title("VIX Volatility Index (yfinance) — Fear Gauge", fontsize=11, fontweight="bold")
ax.legend(fontsize=8, loc="upper right")
ax.annotate("COVID crash\nVIX=82", xy=(pd.Timestamp("2020-03-16"), 82),
            xytext=(pd.Timestamp("2019-06-01"), 70),
            arrowprops=dict(arrowstyle="->", color="#dc2626"), fontsize=9, color="#dc2626")

# SPY
ax = axes[2]
spy_plot = spy.loc[start:]
ax.plot(spy_plot.index, spy_plot["spy"], color="#2563eb", linewidth=1.2)
# Drawdown shading
rolling_max = spy_plot["spy"].expanding().max()
drawdown = (spy_plot["spy"] - rolling_max) / rolling_max * 100
ax.fill_between(spy_plot.index, spy_plot["spy"], rolling_max,
                where=(drawdown < -10), alpha=0.15, color="#dc2626", label=">10% drawdown")
ax.set_ylabel("SPY Price ($)", fontsize=10)
ax.set_title("SPY S&P500 ETF (yfinance) — Risk-on/off Regime", fontsize=11, fontweight="bold")
ax.legend(fontsize=9)

# DXY (UUP proxy)
ax = axes[3]
dxy_plot = dxy.loc[start:]
dxy_ma90 = dxy_plot["dxy"].rolling(90).mean()
ax.plot(dxy_plot.index, dxy_plot["dxy"], color="#6b7280", linewidth=0.8, alpha=0.6, label="UUP (daily)")
ax.plot(dxy_plot.index, dxy_ma90, color="#1d4ed8", linewidth=2, label="90-day MA")
ax.set_ylabel("UUP Price ($)", fontsize=10)
ax.set_title("UUP (DXY Proxy — US Dollar Index) — Dollar Strength vs BTC", fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.annotate("Dollar surge 2022\n→ BTC headwind", xy=(pd.Timestamp("2022-09-01"), 29),
            xytext=(pd.Timestamp("2020-06-01"), 29.5),
            arrowprops=dict(arrowstyle="->", color="#1d4ed8"), fontsize=9, color="#1d4ed8")

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
plt.suptitle("Market Data: VIX + SPY + DXY (yfinance) — Macro Context for BTC DCA",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Signal Engineering — Monetary Policy Signal

Combines Fed rate velocity + M2 growth + DXY momentum into a single [-1, 1] signal.

In [ ]:
from work.model_development import compute_monetary_policy_signal, compute_fear_composite_signal

# Use BTC date index as the base
price_index = btc.loc["2018-01-01":].index

monetary_sig = compute_monetary_policy_signal(price_index)
fear_sig     = compute_fear_composite_signal(price_index)

print(f"Monetary Policy Signal:")
print(f"  Range: [{monetary_sig.min():.3f}, {monetary_sig.max():.3f}]")
print(f"  Mean: {monetary_sig.mean():.3f}, Std: {monetary_sig.std():.3f}")
print(f"  Positive (easy money): {(monetary_sig > 0).sum()} days ({(monetary_sig > 0).mean()*100:.1f}%)")
print(f"  Negative (tight money): {(monetary_sig < 0).sum()} days ({(monetary_sig < 0).mean()*100:.1f}%)")
print()
print(f"Fear Composite Signal:")
print(f"  Range: [{fear_sig.min():.3f}, {fear_sig.max():.3f}]")
print(f"  Mean: {fear_sig.mean():.3f}, Std: {fear_sig.std():.3f}")
print(f"  Positive (fear = buy): {(fear_sig > 0).sum()} days ({(fear_sig > 0).mean()*100:.1f}%)")
print(f"  Negative (greed = reduce): {(fear_sig < 0).sum()} days ({(fear_sig < 0).mean()*100:.1f}%)")

In [ ]:
# Decompose monetary signal into components
dff_full = pd.read_csv(EXT / "fred" / "dff.csv", index_col="date", parse_dates=True)
dff_reindexed = dff_full["dff"].reindex(price_index, method="ffill").ffill().bfill()
rate_change_3m = dff_reindexed.diff(63)
rate_signal = np.tanh(-rate_change_3m / 2.0)

m2_full = pd.read_csv(EXT / "fred" / "m2sl.csv", index_col="date", parse_dates=True)
m2_reindexed = m2_full["m2sl"].reindex(price_index, method="ffill").ffill().bfill()
m2_yoy = m2_reindexed.pct_change(365) * 100
m2_signal = np.tanh(m2_yoy / 15.0)

dxy_full = pd.read_csv(EXT / "market" / "dxy.csv", index_col="date", parse_dates=True)
dxy_reindexed = dxy_full["dxy"].reindex(price_index, method="ffill").ffill().bfill()
dxy_roll_mean = dxy_reindexed.rolling(90, min_periods=30).mean()
dxy_roll_std  = dxy_reindexed.rolling(90, min_periods=30).std().replace(0, np.nan)
dxy_z = ((dxy_reindexed - dxy_roll_mean) / dxy_roll_std).fillna(0).clip(-3, 3)
dxy_signal = np.tanh(-dxy_z * 0.5)

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# Composite
ax = axes[0]
ax.plot(price_index, monetary_sig, color="#d97706", linewidth=2)
ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax.fill_between(price_index, monetary_sig, 0, where=(monetary_sig > 0), alpha=0.2, color="#d97706", label="Easy conditions")
ax.fill_between(price_index, monetary_sig, 0, where=(monetary_sig < 0), alpha=0.2, color="#991b1b", label="Tight conditions")
ax.set_ylabel("Signal", fontsize=10)
ax.set_title("Monetary Policy Signal (COMPOSITE) — Easy=buy more, Tight=buy less", fontsize=11, fontweight="bold")
ax.legend(fontsize=9)

# Fed rate component
ax = axes[1]
ax.plot(price_index, rate_signal * 0.50, color="#dc2626", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax.fill_between(price_index, rate_signal*0.50, 0, where=(rate_signal > 0), alpha=0.15, color="#059669")
ax.fill_between(price_index, rate_signal*0.50, 0, where=(rate_signal < 0), alpha=0.15, color="#dc2626")
ax.set_ylabel("Signal (×0.50 weight)", fontsize=10)
ax.set_title("Component 1: Fed Funds Rate 3-Month Velocity (50% weight) — hike→reduce DCA", fontsize=11, fontweight="bold")

# M2 component
ax = axes[2]
ax.plot(price_index, m2_signal * 0.25, color="#2563eb", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax.fill_between(price_index, m2_signal*0.25, 0, where=(m2_signal > 0), alpha=0.15, color="#2563eb")
ax.fill_between(price_index, m2_signal*0.25, 0, where=(m2_signal < 0), alpha=0.15, color="#dc2626")
ax.set_ylabel("Signal (×0.25 weight)", fontsize=10)
ax.set_title("Component 2: M2 YoY Growth (25% weight) — expansion→buy more", fontsize=11, fontweight="bold")

# DXY component
ax = axes[3]
ax.plot(price_index, dxy_signal * 0.25, color="#7c3aed", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
ax.fill_between(price_index, dxy_signal*0.25, 0, where=(dxy_signal > 0), alpha=0.15, color="#7c3aed")
ax.fill_between(price_index, dxy_signal*0.25, 0, where=(dxy_signal < 0), alpha=0.15, color="#dc2626")
ax.set_ylabel("Signal (×0.25 weight)", fontsize=10)
ax.set_title("Component 3: DXY Momentum (25% weight) — strong dollar→reduce DCA", fontsize=11, fontweight="bold")

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
plt.suptitle("Monetary Policy Signal Decomposition: Fed Rate + M2 + DXY", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Signal Correlation Analysis

In [ ]:
# Load full feature set
from work.model_development import precompute_features

btc_full = load_data()
features = precompute_features(btc_full)
feat = features.loc["2018-01-01":].copy()

# BTC forward returns (30-day)
btc_price = btc_full["PriceUSD_coinmetrics"].loc[feat.index]
feat["fwd_30d"] = btc_price.pct_change(30).shift(-30) * 100

# Correlation of each signal with forward 30-day BTC returns
signal_cols = ["mvrv_zscore", "cycle_signal", "exchange_signal", "monetary_signal", "fear_signal", "price_vs_ma"]
feat_clean = feat[signal_cols + ["fwd_30d"]].dropna()

corr_matrix = feat_clean.corr()
fwd_corr = corr_matrix["fwd_30d"].drop("fwd_30d").sort_values()

print("Correlation of signals with 30-day forward BTC returns:")
for sig, corr in fwd_corr.items():
    direction = "(buy = good)" if corr > 0 else "(sell = good)"
    print(f"  {sig:25s}: {corr:+.4f}  {direction}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Correlation bar chart
ax = axes[0]
colors = ["#059669" if v > 0 else "#dc2626" for v in fwd_corr.values]
bars = ax.barh(fwd_corr.index, fwd_corr.values, color=colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=1)
for bar, val in zip(bars, fwd_corr.values):
    ax.text(val + (0.002 if val >= 0 else -0.002), bar.get_y() + bar.get_height()/2,
            f"{val:+.4f}", va="center", ha="left" if val >= 0 else "right", fontsize=9)
ax.set_xlabel("Pearson Correlation with 30-day Forward BTC Return", fontsize=10)
ax.set_title("Signal → Forward Return Correlation", fontsize=11, fontweight="bold")

# Full correlation heatmap
ax = axes[1]
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(corr_matrix, ax=ax, cmap="RdYlGn", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", square=True, linewidths=0.5,
            annot_kws={"size": 9})
ax.set_title("Signal Correlation Matrix", fontsize=11, fontweight="bold")

plt.suptitle("Signal Quality: Correlation with 30-day Forward BTC Returns",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Quintile analysis: does signal predict future returns?
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

signal_labels = {
    "mvrv_zscore":    "MVRV Z-score (inverted = buy low)",
    "cycle_signal":   "Halving Cycle Signal [NOVEL]",
    "exchange_signal":"Exchange Flow Signal [NOVEL]",
    "monetary_signal":"Monetary Policy Signal [NOVEL]",
    "fear_signal":    "Fear Composite Signal [NOVEL]",
    "price_vs_ma":    "Price vs 200-day MA",
}

for ax, (col, label) in zip(axes, signal_labels.items()):
    valid = feat_clean[[col, "fwd_30d"]].dropna()
    valid["quintile"] = pd.qcut(valid[col], q=5, labels=["Q1\n(Lowest)", "Q2", "Q3", "Q4", "Q5\n(Highest)"])
    q_means = valid.groupby("quintile", observed=True)["fwd_30d"].mean()

    colors_q = ["#059669" if v > 0 else "#dc2626" for v in q_means.values]
    ax.bar(range(len(q_means)), q_means.values, color=colors_q, alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
    ax.set_xticks(range(len(q_means)))
    ax.set_xticklabels(q_means.index, fontsize=8)
    ax.set_ylabel("Avg 30d Fwd Return (%)", fontsize=8)
    ax.set_title(label, fontsize=8, fontweight="bold")

plt.suptitle("Quintile Analysis: Signal Quintile → Average 30-day Forward BTC Return\n"
             "(Monotonic pattern = signal has predictive power)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - cycle_signal: Q5 (strong buy signal) → should have higher fwd returns than Q1")
print("  - mvrv_zscore: Q1 (low MVRV = undervalued) → should have higher fwd returns than Q5")
print("  - monetary_signal: Q5 (easy money) → should have higher fwd returns than Q1 (tight)")

## 6. Model Integration — Backtest Results

In [ ]:
import json

with open(ROOT / "work/output/metrics.json") as f:
    m = json.load(f)
with open(ROOT / "example_1/output/metrics.json") as f:
    m_ex = json.load(f)

our  = m["summary_metrics"]
ex1  = m_ex["summary_metrics"]

print("=" * 65)
print("BACKTEST RESULTS (2018 ~ 2025, rolling 1-year windows)")
print("=" * 65)
print(f"{'Metric':<30} {'Our Model':>15} {'Example 1':>12} {'Delta':>10}")
print("-" * 65)
print(f"{'Win Rate':<30} {our['win_rate']:>14.2f}% {ex1['win_rate']:>11.2f}% {our['win_rate']-ex1['win_rate']:>+9.2f}%p")
print(f"{'Model Score':<30} {our['score']:>14.2f}% {ex1['score']:>11.2f}% {our['score']-ex1['score']:>+9.2f}%p")
print(f"{'Mean Excess Percentile':<30} {our['mean_excess']:>14.2f}% {ex1['mean_excess']:>11.2f}% {our['mean_excess']-ex1['mean_excess']:>+9.2f}%p")
print(f"{'Relative Improvement':<30} {our['relative_improvement_pct_mean']:>14.2f}% {ex1['relative_improvement_pct_mean']:>11.2f}% {our['relative_improvement_pct_mean']-ex1['relative_improvement_pct_mean']:>+9.2f}%p")
print(f"{'Sats/$ Ratio':<30} {our['mean_ratio']:>15.3f} {ex1['mean_ratio']:>12.3f} {our['mean_ratio']-ex1['mean_ratio']:>+10.3f}")
print(f"{'Wins / Total Windows':<30} {our['wins']:>5}/{our['total_windows']} {ex1['wins']:>5}/{ex1['total_windows']}")
print("=" * 65)

In [ ]:
# Year-by-year breakdown
df_our = pd.DataFrame(m["window_level_data"])
df_our["start_date"] = pd.to_datetime(df_our["start_date"])
df_our["year"] = df_our["start_date"].dt.year
df_our["excess"] = df_our["dynamic_percentile"] - df_our["uniform_percentile"]
df_our["wins"] = df_our["excess"] > 0

df_ex = pd.DataFrame(m_ex["window_level_data"])
df_ex["start_date"] = pd.to_datetime(df_ex["start_date"])
df_ex["year"] = df_ex["start_date"].dt.year
df_ex["excess"] = df_ex["dynamic_percentile"] - df_ex["uniform_percentile"]
df_ex["wins"] = df_ex["excess"] > 0

by_year_our = df_our.groupby("year").agg(win_rate=("wins", "mean"), mean_excess=("excess", "mean"))
by_year_ex  = df_ex.groupby("year").agg(win_rate=("wins", "mean"), mean_excess=("excess", "mean"))

print(f"{'Year':<6} {'WR (Ours)':>10} {'WR (Ex1)':>9} {'Δ WR':>7} {'Exc (Ours)':>11} {'Exc (Ex1)':>10} {'Δ Exc':>8}")
print("-" * 65)
for year in sorted(set(by_year_our.index) | set(by_year_ex.index)):
    wr_our = by_year_our.loc[year, "win_rate"]*100 if year in by_year_our.index else 0
    wr_ex  = by_year_ex.loc[year, "win_rate"]*100  if year in by_year_ex.index  else 0
    exc_our = by_year_our.loc[year, "mean_excess"] if year in by_year_our.index else 0
    exc_ex  = by_year_ex.loc[year, "mean_excess"]  if year in by_year_ex.index  else 0
    print(f"  {year}  {wr_our:>8.1f}%  {wr_ex:>7.1f}%  {wr_our-wr_ex:>+6.1f}%  {exc_our:>+9.2f}%  {exc_ex:>+8.2f}%  {exc_our-exc_ex:>+7.2f}%")

## 7. Summary

### What the External Data Contributes

| Signal | Key Period | Effect |
|--------|-----------|--------|
| **Monetary (FRED)** | 2022 rate hike cycle | Correctly reduced DCA weight as Fed raised rates 0→5.33%. Improved 2022 win rate by ~8%p vs example_1. |
| **M2 YoY (FRED)** | 2020-2021 | Positive signal during COVID stimulus expansion, supporting accumulation before the bull market. |
| **DXY (yfinance)** | 2022 Sep peak | Dollar strength peaked Sept 2022 — correct negative signal during BTC bottom. |
| **VIX (yfinance)** | COVID crash | VIX hit 82 in Mar 2020 — strong contrarian buy signal at exact bottom. |
| **Fear & Greed** | 2022 Nov lows | Score=6 (Extreme Fear) during $16k bottom — cubic transform gave strong buy signal. |

### Design Decisions
1. **Cubic transformation** for Fear & Greed: prevents over-suppressing buying at moderate greed (FG=75 → -0.13 vs linear -0.50)
2. **3-month velocity** for Fed rate: captures pace of change, not absolute level — a 5% rate that's been stable for a year is less damaging than rapid hikes
3. **Cap at -0.3** for monetary signal during deep MVRV value zones: prevents double-penalizing when asset is already at historic lows
4. **FRED via pandas_datareader**: no API key required, production-safe for any user
5. **yfinance with Alpaca fallback**: ensures data availability; alternative.me free API needs no authentication